# A Minimal Neural Network
### Classifying Iris Flowers

In this notebook we build and train a small neural network from scratch.

**Goal:** predict the species of an iris flower from four measurements.

**What you will see:**
1. Load data
2. Define a network
3. Train with gradient descent
4. Evaluate accuracy

No heavy abstractions — every step is visible.

## 1. Imports

In [ ]:
import torch
import torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

## 2. The Data

The **Iris dataset** has 150 flowers, each described by 4 numbers:
- sepal length, sepal width, petal length, petal width

Each flower belongs to one of 3 species (classes 0, 1, 2).

In [ ]:
# Load the dataset
iris = load_iris()
X = iris.data    # shape: (150, 4)  -- 150 flowers, 4 features each
y = iris.target  # shape: (150,)    -- the correct class for each flower

print("Features shape:", X.shape)
print("Labels shape: ", y.shape)
print("Classes:      ", iris.target_names)

In [ ]:
# Split into training (80%) and test (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test,  dtype=torch.long)

print("Training samples:", X_train.shape[0])
print("Test samples:    ", X_test.shape[0])

## 3. The Network

Our network has three layers:

```
Input (4)  -->  Hidden (16)  -->  Output (3)
```

- **Input layer:** one node per feature (4 features)
- **Hidden layer:** 16 nodes with ReLU activation  
  *(ReLU simply zeros out negative values: `max(0, x)`)*
- **Output layer:** one node per class (3 classes)

The output layer produces raw scores (logits). The highest score is the predicted class.

In [ ]:
class IrisNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4, 16)   # 4 inputs  -> 16 hidden nodes
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(16, 3)   # 16 hidden -> 3 output scores

    def forward(self, x):
        x = self.layer1(x)   # linear transformation
        x = self.relu(x)     # non-linearity
        x = self.layer2(x)   # produces 3 scores (one per class)
        return x

model = IrisNet()
print(model)

## 4. Loss and Optimizer

**Loss function:** Cross-Entropy Loss  
Measures how wrong the network's predictions are. Lower is better.

**Optimizer:** SGD (Stochastic Gradient Descent)  
Updates the network's weights to reduce the loss.

The **learning rate** controls how large each update step is.

In [ ]:
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

## 5. Training

One **epoch** = one full pass over all training data.

Each step:
1. **Forward pass** — compute predictions
2. **Compute loss** — how wrong are we?
3. **Backward pass** — compute gradients
4. **Update weights** — take one SGD step

In [ ]:
num_epochs = 200
loss_history = []  # we will plot this later

for epoch in range(num_epochs):
    # --- Step 1: Forward pass ---
    predictions = model(X_train)          # shape: (120, 3)

    # --- Step 2: Compute loss ---
    loss = loss_fn(predictions, y_train)

    # --- Step 3: Backward pass ---
    optimizer.zero_grad()   # clear old gradients
    loss.backward()         # compute new gradients

    # --- Step 4: Update weights ---
    optimizer.step()

    loss_history.append(loss.item())

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d} / {num_epochs}  |  Loss: {loss.item():.4f}")

## 6. Learning Curve

The loss should decrease over time — the network is learning.

In [ ]:
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss over Time")
plt.grid(True)
plt.show()

## 7. Evaluation

Now we test on the held-out data the model has **never seen**.

In [ ]:
with torch.no_grad():   # no gradient computation needed for evaluation
    test_scores = model(X_test)           # shape: (30, 3)
    predicted   = test_scores.argmax(1)   # pick the class with highest score

correct  = (predicted == y_test).sum().item()
total    = y_test.shape[0]
accuracy = correct / total * 100

print(f"Correct: {correct} / {total}")
print(f"Accuracy: {accuracy:.1f}%")

## 8. One Prediction Up Close

Let's look at what the network actually produces for a single flower.

In [ ]:
flower = X_test[0].unsqueeze(0)   # shape: (1, 4)
true_label = y_test[0].item()

with torch.no_grad():
    scores = model(flower)         # raw scores, shape: (1, 3)

# Convert scores to probabilities with softmax
probs = torch.softmax(scores, dim=1)[0]

print("True species:   ", iris.target_names[true_label])
print()
for i, name in enumerate(iris.target_names):
    print(f"  P({name:15s}) = {probs[i].item():.3f}")

---
## Recap

| Step | What we did |
|------|-------------|
| Data | Loaded Iris, split into train/test |
| Model | 2-layer network: Linear → ReLU → Linear |
| Loss | Cross-entropy (for multi-class classification) |
| Training | 200 epochs of SGD |
| Evaluation | Accuracy on held-out test set |

**Things to try:**
- Change `lr` (e.g. `0.1` or `0.01`) — what happens to training?
- Change the hidden layer size (`16` → `4` or `64`)
- Add a second hidden layer
- Train for more or fewer epochs